# Cross integration

Cross integration combines batches that all measure the same modalities. The task is to remove batch effects and keep the biological structure. Every cross method here reads RNA and ADT; for several 10x Multiome samples, use the vertical tutorial. This tutorial uses `D52`, with 23,478 cells in three batches.

## 1. Install

The first cell installs `multibench-sc`. Methods run only on Linux, each in its own environment. Set `INSTALL_ENVS = True` on Linux or Colab to download the environments. On macOS and Windows every cell still runs and uses stored outputs instead.

<details>
<summary>Details</summary>

With `INSTALL_ENVS = False`, the notebook downloads only the reference data and the stored outputs. A method cell then prints one line and loads those outputs.

With `INSTALL_ENVS = True`, the cells download prebuilt environments and run the methods, with no conda needed.

On Colab, choose a GPU runtime first: Runtime -> Change runtime type -> T4 GPU. On a CPU runtime, training methods are much slower.

</details>

In [ ]:
import importlib.metadata, importlib.util, sys
if importlib.util.find_spec("multibench") is None:
    # keep the numpy / pandas this interpreter already has
    pins = [f"{p}=={importlib.metadata.version(p)}" for p in ("numpy", "pandas") if importlib.util.find_spec(p)]
    !{sys.executable} -m pip -q install "multibench-sc>=0.3.2" {" ".join(pins)}
    importlib.invalidate_caches()
    # not on PyPI yet: install from GitHub
    if importlib.util.find_spec("multibench") is None:
        !{sys.executable} -m pip -q install "git+https://github.com/DSichang/scMultiBench.git" {" ".join(pins)}
else:
    print("multibench already installed")

In [ ]:
# False: no environment is downloaded; method cells use stored outputs.
# True (Linux or Colab): run the methods here; no conda needed.
# 2 envs, 1.4 GB to download on a CPU host, 3.0 GB on a GPU host.
INSTALL_ENVS = False

In [ ]:
%matplotlib inline
import warnings
from pathlib import Path
import anndata
import pandas as pd
from tqdm import TqdmWarning
# hide library warnings; the warnings of multibench stay visible
for _w in (FutureWarning, DeprecationWarning, pd.errors.PerformanceWarning,
           anndata.ImplicitModificationWarning, TqdmWarning):
    warnings.filterwarnings("ignore", category=_w)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
import multibench as mtb

DATASET  = "D52"
CATEGORY = "cross"
# the reference data (179 MB), downloaded once
mtb.data.fetch('D52')
print("multibench", mtb.__version__)

## 2. Run the analysis

### Environments: only if you will run methods

The next cell installs the environments for UINMF, sciPENN and StabMap: 2 envs, 1.4 GB to download on a CPU host, 3.0 GB on a GPU host. It downloads nothing unless `INSTALL_ENVS = True` and the computer runs Linux.

<details>
<summary>Details</summary>

Environments are unpacked under `mtb.config.DEFAULT.envs_dir`. Set it before this cell to use another disk. Without conda on the computer, the default is `~/.cache/multibench/envs`. An environment that is already there is not downloaded again.

To install every cross environment from a terminal, run `multibench env install --category cross --packed --run`. That is 7 envs, 11.1 GB to download on a CPU host, 17.0 GB on a GPU host; `multibench env plan --category cross` lists the size of each.

</details>

In [ ]:
import sys
if not INSTALL_ENVS:
    print("INSTALL_ENVS is False: no environment is downloaded")
elif sys.platform != "linux":
    print("method environments run only on Linux: skipped on", sys.platform)
else:
    # dry run: sizes only
    plan = mtb.env.install(['UINMF', 'sciPENN', 'StabMap'], category=CATEGORY)
    todo = [r for r in plan if not r["exists"]]
    print(f"{len(todo)} of {len(plan)} envs to download, {sum(r['archive_bytes'] or 0 for r in todo) / 1e9:.1f} GB")
    for r in mtb.env.install(['UINMF', 'sciPENN', 'StabMap'], category=CATEGORY, packed=True, dry_run=False):
        print(f"{r['env']:20s} {r['state']}")

### Run the methods

`run_all` runs UINMF, sciPENN and StabMap on `D52`, each in its own environment. Each method writes an embedding. An embedding is a table of numbers with one row per cell. `run_all` scores each embedding with the scIB metrics. Without environments, the cell prints one line and loads stored outputs instead.

<details>
<summary>Details</summary>

The stored outputs are the `run_all` outputs for `D52` from the benchmark's Linux machine. `mtb.data.fetch_outputs` downloads them and `mtb.load_batch` reloads them, so the embeddings and run times are real.

If that download fails, the cell uses the stored metric table (`load_results(source="rerun")`) instead. The `status` column then reads `STORED`, and there is no embedding to score.

UINMF reads only batches 1 and 2 of 3, and `scan` says so in its `caveat` column. Its `emb_shape` counts fewer cells, and its metrics cover only those cells.

</details>

In [ ]:
def stored_sweep(dataset, methods=None):
    """The stored results for `dataset`, as the object `run_all` returns."""
    long = mtb.load_results(CATEGORY, dataset=dataset, source="rerun", methods=methods)
    recs = [{"method": m, "status": "STORED", "metrics": g.set_index("metric")["value"].to_dict()}
            for m, g in long.groupby("method")]
    return mtb.BatchResult(recs, dataset, CATEGORY)

def replacement(dataset, methods, stored):
    """The run_all outputs for `dataset` from the benchmark's Linux machine; the stored results if that download fails."""
    try:
        res = mtb.load_batch(mtb.data.fetch_outputs(dataset), methods=methods)
        print(f"replacement: the run_all outputs for {dataset} from the benchmark's Linux machine, with embeddings and run times")
        return res
    # offline, or the outputs are not published yet
    except Exception as e:
        print(f"replacement: the package's stored metric table. {type(e).__name__} from fetch_outputs: {e}")
        return stored_sweep(*stored)

check = mtb.scan("D52", CATEGORY, methods=['UINMF', 'sciPENN', 'StabMap'])
if check.env_ok.any():
    res = mtb.run_all("D52", CATEGORY,
                      methods=['UINMF', 'sciPENN', 'StabMap'],
                      out_dir="/tmp/tutorial_cross")
else:
    print("no method environment on this computer: the run is skipped, and outputs computed elsewhere replace it")
    res = replacement("D52", ['UINMF', 'sciPENN', 'StabMap'], stored=("D52", ['UINMF', 'sciPENN', 'StabMap']))
res.summary

### Score one embedding

`run_all` has already scored every method. To score one embedding yourself, pass the file the method wrote and the dataset's label files to `mtb.evaluate`:

<details>
<summary>Details</summary>

`label_order=` gives the order in which the method stacked its cells. The cell takes it from the run record's `labels_used`.

`metrics=` selects what is computed:

- `None` (the default): every applicable metric
- `"clustering"`, `"batch"` or `"all"`: a family
- a list such as `["ARI", "NMI"]`: those metrics

Batch metrics need `batch=` or several label files.

The default Leiden backend is igraph. To compare your scores with the stored tables, set `mtb.config.DEFAULT.leiden_flavor = "leidenalg"` before `evaluate`. The two backends can move ARI by up to about 0.1.

</details>

In [ ]:
m, emb = "StabMap", None
if res.out_dir is not None:
    emb = Path(res.out_dir) / f"{m}_{res.dataset}" / "embedding.h5"
if emb is None or not emb.is_file():
    print(f"no embedding on this computer for {m}: nothing to score")
    scores = None
else:
    rec = next(r for r in res.results if r["method"] == m)
    # the label files in the order the method stacked its cells
    order = [Path(f).stem for f in rec.get("labels_used") or []] or None
    scores = mtb.evaluate(emb, labels=mtb.labels_for(res.dataset), label_order=order, verbose=False)
scores.T if scores is not None else None

### Plot

`res.plot()` draws a bubble table. Circle size shows the rank within a column; bigger is better. The fill compares the value with the other rows in the same column: the lightest fill is the lowest value in this figure, not zero.

<details>
<summary>Details</summary>

Metrics are grouped by family: blue for dimension reduction and clustering, green for batch correction. Each family starts with an `Overall` bar. Its length and colour both show the family score.

A column whose rows all hold the same value is drawn grey, and the note under the figure names it. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 3. Your own data

The same calls work on a folder of your own data. Give raw counts for every modality, as in the demo data; the methods normalise the data themselves. `describe_layout` prints the files a cross dataset needs:

In [ ]:
print(mtb.describe_layout(CATEGORY))

`labels_for` returns a dataset's label files: `cty1`, `cty2`, ... in that order. Some methods stack their cells in another order. `labels_for(DATASET, CATEGORY, method)` returns the files in that method's order. A wrong order gives wrong scores without an error.

<details>
<summary>Details: label order</summary>

On `D52`, `labels_for` returns another order for StabMap (`cty3, cty1, cty2`).

A method that reads only some batches gets only their label files: `cty1, cty2` for UINMF.

Pass the dict that `labels_for` returns to `evaluate` unchanged. A dict you build or reorder yourself must be in the default order. For any other order, name the keys with `label_order=`.

`run_all` scores every order that fits the cell count and keeps the one with the highest ARI. The `label_order` column of `res.summary` shows the order it kept.

</details>

In [ ]:
labels = mtb.labels_for(DATASET)            # {file stem: path}
print({k: Path(v).name for k, v in labels.items()})
print(*Path(next(iter(labels.values()))).read_text().splitlines()[:4], sep="\n")
print("StabMap:", list(mtb.labels_for(DATASET, CATEGORY, "StabMap")))

`mtb.io.export_dataset` with `batch=` splits one AnnData into numbered files, one set per batch. Here on a synthetic AnnData:

<details>
<summary>Details: export</summary>

For one file per batch, call `export_dataset` once per file with `batch_index=N` instead of `batch=`. `describe_layout` above prints the command.

A call that would replace a file already in the folder raises `FileExistsError`. Pass `overwrite=True` to replace it.

</details>

In [ ]:
import anndata as ad, numpy as np, tempfile, os
rng = np.random.default_rng(0)
# RNA as raw counts, cells x genes
demo = ad.AnnData(X=rng.poisson(1.0, size=(150, 40)).astype(float))
demo.var_names = [f"gene{i}" for i in range(40)]
demo.obsm["protein"] = rng.poisson(3.0, size=(150, 12)).astype(float)
demo.uns["protein_names"] = [f"CD{i}" for i in range(12)]
demo.obs["celltype"] = rng.choice(["T", "B", "NK"], 150)
demo.obs["batch"] = rng.choice(["donor1", "donor2", "donor3"], 150)

folder = mtb.io.export_dataset(demo, os.path.join(tempfile.mkdtemp(), "MYCROSS"),
                               rna="X", adt="obsm:protein", labels="obs:celltype",
                               batch="obs:batch", category="cross")
print(sorted(os.listdir(folder)))
sc = mtb.scan("MYCROSS", CATEGORY, data_path=folder.parent)
sc[["method", "modalities", "files_ok"]]

Next, a real dataset under a new name: a random 60% of `D52`'s cells, with at most 2,000 cells and 5,000 features per file.

<details>
<summary>Details: cell alignment</summary>

Files with the same number of cells keep the same cells in the same order. Each modality file then stays aligned with its label file. An export of your own data must keep this alignment too.

</details>

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd

def subsample_dataset(src_dir, dst_dir, frac=0.6, seed=0, max_cells=2000, max_features=5000):
    """Copy a dataset under a new name, keeping a random `frac` of its cells; files with the same cell count keep the same cells."""
    rng = np.random.default_rng(seed)
    os.makedirs(dst_dir, exist_ok=True)
    counts, keep = {}, {}
    for fn in sorted(os.listdir(src_dir)):
        p = os.path.join(src_dir, fn)
        if fn.endswith(".h5"):
            with h5py.File(p) as f:
                if "matrix/data" in f:
                    # matrix/data is features x cells
                    counts[fn] = f["matrix/data"].shape[1]
        elif fn.endswith(".csv"):
            counts[fn] = len(pd.read_csv(p))
    for n in set(counts.values()):
        k = min(max(50, int(n * frac)), max_cells)
        keep[n] = np.sort(rng.choice(n, size=k, replace=False))
    for fn, n in counts.items():
        sp, dp = os.path.join(src_dir, fn), os.path.join(dst_dir, fn)
        idx = keep[n]
        if fn.endswith(".csv"):
            pd.read_csv(sp).iloc[idx].to_csv(dp, index=False)
        else:
            with h5py.File(sp) as f, h5py.File(dp, "w") as g:
                grp = g.create_group("matrix")
                n_feat = f["matrix/data"].shape[0]
                fidx = np.arange(n_feat) if n_feat <= max_features else np.sort(rng.choice(n_feat, size=max_features, replace=False))
                block = f["matrix/data"][fidx, :] if n_feat > max_features else f["matrix/data"][()]
                grp.create_dataset("data", data=np.asarray(block)[:, idx])
                if "matrix/features" in f:
                    grp.create_dataset("features", data=np.asarray(f["matrix/features"])[fidx])
                if "matrix/barcodes" in f:
                    grp.create_dataset("barcodes", data=np.asarray(f["matrix/barcodes"])[idx])
    return dst_dir

`scan` checks each method variant. A variant is one set of input files that a method accepts. `files_ok` checks the folder and works on any computer. `env_ok` checks the environment. A variant is `runnable` only when both pass, and `reason` says what failed.

In [ ]:
DATA_ROOT = "/tmp/mydata"
src = mtb.config.DEFAULT.data_path / "D52"
subsample_dataset(src, f"{DATA_ROOT}/MYDATA_cross", frac=0.6)

sc = mtb.scan(f"MYDATA_cross", category=CATEGORY, data_path=DATA_ROOT)
print(f"files_ok {int(sc.files_ok.sum())}, env_ok {int(sc.env_ok.sum())}, runnable {int(sc.runnable.sum())} of {len(sc)} method variants")
sc[["method", "modalities", "files_ok", "env_ok", "runnable", "reason"]].head(6)

In [ ]:
if sc[sc.method.isin(['UINMF', 'sciPENN', 'StabMap'])].env_ok.any():
    mine = mtb.run_all(f"MYDATA_cross", CATEGORY,
                       methods=['UINMF', 'sciPENN', 'StabMap'],
                       out_dir=f"{DATA_ROOT}/out_cross",
                       data_path=DATA_ROOT)
else:
    print("no method environment on this computer: the run is skipped, and outputs computed elsewhere replace it")
    # the stored results for D52s, a 60% subsample of D52
    mine = stored_sweep("D52s", ['UINMF', 'sciPENN', 'StabMap'])
mine.summary

In [ ]:
mine.plot()

## 4. Stored results

The package ships stored results for 8 methods on `D52`. `load_results(..., source="rerun")` reads them as a long table, and `mtb.plot.bubble` draws it. A long table has one row per method and metric. Nothing is run.

<details>
<summary>Details: sources</summary>

For `D52`, the published table holds 1 method and the package's own runs hold 8. `load_results` defaults to `source="published"`, so every call here names its source. Where both hold a method, the values can differ.

`run_all` saves its results in `out_dir`, and `mtb.load_batch(out_dir)` reloads them later without running anything. Use one `out_dir` per dataset and category: a second `run_all` into the same folder adds its methods to the saved results, and one for another dataset raises `ValueError`.

</details>

In [ ]:
long = mtb.load_results(CATEGORY, dataset=DATASET, source="rerun")
print(long.method.nunique(), "methods,", long.source.unique())
fig = mtb.plot.bubble(long)
fig.set_dpi(110)
fig

A summary compares methods scored on the same datasets. `aggregate="summary"` ranks the methods over `D52` and `D52s`, a random 60% subsample of `D52`'s cells. `require_complete=True` keeps only the methods with results on both.

<details>
<summary>Details: summary bars</summary>

Each metric bar is the method's rank averaged over the datasets, then min-max scaled. `Overall` summarises the ranks of the family's metrics. Bar length and colour both show the value.

Without `require_complete=True`, a method that is missing from one dataset gets the lowest rank there, which pulls its bars down.

`D52s` is a random subsample used for the package's second run of the methods. It cannot be downloaded or rebuilt, so a summary that includes your own method uses the full datasets only.

</details>

In [ ]:
pair = mtb.load_results(CATEGORY, dataset=[DATASET, DATASET + "s"], source="rerun")
print(pair.groupby("dataset").method.nunique().to_dict())
mtb.plot.bubble(pair, aggregate="summary", require_complete=True,
                title=f"Summary of 2 cross datasets")

`results_coverage` counts the methods each stored source holds for this dataset:

In [ ]:
cov = mtb.results_coverage(CATEGORY)
cov[cov.dataset == DATASET].groupby("source").method.nunique()

## 5. Reference

### What runs on a dataset, and why not

`scan` on `D52` runs nothing. The first table lists the variants whose input files are in place, and `caveat` says what to check before a run. The second table says why the other variants do not fit.

<details>
<summary>Details</summary>

For a missing environment, `env_reason` gives the install command, or says that the environment runs only on Linux.

From a terminal, `multibench scan D52 --category cross` prints the same scan. `--columns all` adds every column, including `command`: the exact command `run` would execute.

</details>

In [ ]:
avail = mtb.scan(DATASET, category=CATEGORY)
print(f"files_ok {int(avail.files_ok.sum())}, env_ok {int(avail.env_ok.sum())}, runnable {int(avail.runnable.sum())} of {len(avail)} method variants")
avail[avail.files_ok][["method", "modalities", "env", "env_ok", "env_reason",
                       "output_kind", "needs_labels", "runtime_tier", "caveat"]]

In [ ]:
not_ok = avail[~avail.files_ok][["method", "modalities", "files_reason"]]
not_ok.head(5) if len(not_ok) else "(every method's inputs resolve on this dataset)"


### Tuning

The table counts the parameters each variant exposes. `mtb.params_for(method, CATEGORY, modalities)` lists them, and `run_all(..., params={"Method": {"key": value}})` sets them.

<details>
<summary>Details</summary>

Many upstream scripts set their parameters in code. Their variants accept no `params`.

From a terminal, `multibench params METHOD` prints the table, and `multibench run-all ... --param METHOD:KEY=VALUE` sets a value.

</details>

In [ ]:
rows = [{"method": m, "modalities": "+".join(v["modalities"]) or "(data_dir)",
         "n_tunable": v["n_tunable"], "needs_labels": v["needs_labels"],
         "output_kind": v["output_kind"]}
        for m in sorted(mtb.list_methods(category=CATEGORY))
        for v in mtb.method_info(m)["supports"] if v["category"] == CATEGORY]
pd.DataFrame(rows).sort_values(["n_tunable", "method"], ascending=[False, True]).reset_index(drop=True)

### A method's record and citation

`method_info` returns what the package knows about a method, including its reference and repository. `mtb.cite` returns the citations for the benchmark and for the methods you pass. The cell cites StabMap; for your own work, pass every method you ran.

<details>
<summary>Details</summary>

`needs_labels` is True when any variant of the method needs cell-type labels. Each entry of `supports` gives it per variant, together with the modalities and the output kind.

`gpu` says how the method uses a GPU: `required`, `used when present`, `not used` or `unknown`.

`verbose=True` adds the long notes.

</details>

In [ ]:
info = mtb.method_info("StabMap", verbose=True)
{k: info[k] for k in ("id", "env", "needs_labels", "atac", "gpu", "notes", "repo_url", "version", "reference")}

In [ ]:
print(mtb.cite(["StabMap"]))   # fmt="bibtex" for BibTeX entries

### The metrics

There are two families, and higher is better for every metric.

| family | metrics | measures |
|---|---|---|
| clustering / bio-conservation | `ARI`, `NMI`, `ASW`, `iASW`, `iF1`, `cLISI` | whether the embedding separates the annotated cell types |
| batch correction | `ASW_batch`, `GC`, `iLISI` (+ opt-in `kBET`) | whether the batches mix within each cell type |

<details>
<summary>Details</summary>

ARI can fall slightly below 0; about 0 means a random clustering. Every other metric lies between 0 and 1. `mtb.catalog.metrics()` describes each one.

Batch metrics appear only when the dataset has more than one batch.

kBET is computed only when named, as in `metrics=["ASW_batch", "GC", "iLISI", "kBET"]`. It is much slower than the others.

</details>

### Methods from the benchmark study

The cell compares the methods the scMultiBench study benchmarked for cross integration with the methods this package has a cross variant for. It prints each missing method with the categories it has variants for.

In [ ]:
# benchmarked for cross on dimension reduction, batch correction and clustering
paper = ['totalVI', 'scMoMaT', 'UnitedNet', 'sciPENN', 'Concerto', 'scMDC', 'StabMap', 'UINMF', 'scMM', 'MOFA2', 'Multigrate']
in_package = set(mtb.list_methods())
has_variant = sorted(m for m in in_package
                     if any(v["category"] == CATEGORY for v in mtb.method_info(m)["supports"]))
missing = [m for m in paper if m not in has_variant]
print(f"the study benchmarks {len(paper)} {CATEGORY} methods on dimension reduction, batch correction and clustering; this package has a {CATEGORY} variant for {len(has_variant)}")
for m in missing:
    if m in in_package:
        print(f"  {m}: variants for {', '.join(mtb.method_info(m)['categories'])} only")
    else:
        print(f"  {m}: not in this package")
if not missing:
    print("every benchmarked method has a variant for this category")

## Troubleshooting

`scan`'s `reason` column says why a method is not runnable. `res.failures` says why a run failed or a method you named was skipped.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| `files_ok` False: input files not found | `reason` names the missing file |
| `env_ok` False on Linux | run the `multibench env install ...` command in `env_reason` |
| `env_ok` False on macOS or Windows | methods run only on Linux: run the same calls there |
| `env_ok` False: the method needs an NVIDIA GPU | run it on a GPU machine; `mtb.scan(..., assume_gpu=True)` checks everything else on a computer without one |
| `FileExistsError` from `export_dataset` | the folder already holds the file: pass `overwrite=True` to replace it |
| a warning that values are not whole numbers | export raw counts, for example with `rna="layer:counts"` |
| `... which is cells x features` | the matrix is transposed: export it again with `mtb.io.export_dataset` or `mtb.io.to_canonical` |
| a method fails | `res.failures.iloc[0]["error"]` ends with the method's stderr |
| a method times out | raise `timeout=` in `run_all` |
| low `label_order_confidence` | several label files fit the cell count: check `label_order_candidates` in `res.results` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- the other tutorials: vertical, diagonal, mosaic
- the [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/): the full benchmark's rankings, no install needed
- `mtb.recommend(CATEGORY, modalities=[...])`: a ranking of methods from the stored results, to read as a hint
- `mtb.sweep(...)`: one method over a range of values of one parameter